<a href="https://colab.research.google.com/github/Shuvo1523004/WYDOT-HR-Chatbot/blob/main/WYDOT_Verification_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import json
import os
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Optional plotting. The script will still run if matplotlib is unavailable.
try:
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover
    plt = None

# Optional XGBoost. If not installed, the script falls back to sklearn GradientBoosting.
try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    HAS_XGBOOST = False

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# -----------------------------------------------------------------------------
# 1. File paths
# -----------------------------------------------------------------------------
BASE_DIR = Path("/mnt/data") if Path("/mnt/data").exists() else Path.cwd()
LEAVER_XLSX = BASE_DIR / "Exit survey data.xlsx"
STAYER_CSV = BASE_DIR / "anylogic_item_based_hypothetical_pulse_survey.csv"
OUTPUT_DIR = BASE_DIR / "wydot_model_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20

# -----------------------------------------------------------------------------
# 2. Standard 32 feature names used by the HTML tool
# -----------------------------------------------------------------------------
FEATURES: List[str] = [
    "coop_dept",
    "coop_other",
    "team",
    "valued",
    "pay",
    "career",
    "pmi",
    "recognition",
    "sup_feedback",
    "sup_suggest",
    "sup_comfort",
    "complaints",
    "sup_sensitive",
    "sup_comm",
    "upper_mgmt",
    "job_training",
    "training_prog",
    "proper_training",
    "resources",
    "phys_conditions",
    "clarity",
    "get_resources",
    "safety_culture",
    "safety_sup",
    "job_security",
    "informed",
    "sup_policy",
    "hr_comfort",
    "fairness",
    "sup_teamwork",
    "resolve_problems",
    "supervisor_environment_safety",  # used as safety_sup fallback; kept separate only if present
]

# The HTML form has 31 visible fields in some versions because safety_sup and
# supervisor_environment_safety may be treated as the same. To preserve the 32-item
# manuscript structure, we use safety_sup as the final field and remove duplicate later.
FEATURES = [
    "coop_dept", "coop_other", "job_training", "resources", "pmi", "training_prog",
    "pay", "career", "phys_conditions", "get_resources", "valued", "sup_feedback",
    "sup_suggest", "proper_training", "complaints", "sup_comfort", "upper_mgmt",
    "hr_comfort", "informed", "job_security", "team", "safety_culture", "clarity",
    "recognition", "resolve_problems", "sup_sensitive", "sup_feedback_performance",
    "sup_comm", "sup_policy", "fairness", "sup_teamwork", "safety_sup"
]

THEME_GROUPS: Dict[str, List[str]] = {
    "Work Culture and Team": ["coop_dept", "coop_other", "team", "valued"],
    "Compensation and Career": ["pay", "career", "pmi", "recognition"],
    "Supervisor/Leadership": [
        "sup_feedback", "sup_suggest", "sup_comfort", "complaints",
        "sup_sensitive", "sup_comm", "upper_mgmt", "sup_feedback_performance"
    ],
    "Training and Development": ["job_training", "training_prog", "proper_training"],
    "Working Conditions and Resources": ["resources", "phys_conditions", "clarity", "get_resources"],
    "Safety and Job Security": ["safety_culture", "safety_sup", "job_security"],
    "Communication and Information": ["informed", "sup_policy", "hr_comfort"],
    "HR/Fairness/Problem Resolution": ["fairness", "sup_teamwork", "resolve_problems"],
}

# -----------------------------------------------------------------------------
# 3. Manual leaver map, if automatic matching fails
# -----------------------------------------------------------------------------
# Example:
# MANUAL_LEAVER_MAP = {
#     "coop_dept": "Cooperation within your department/program",
#     "pay": "Rate of pay for your job",
# }
MANUAL_LEAVER_MAP: Dict[str, str] = {
    "sup_comfort": "IfIhadquestionsorconcernsIfeltcomfortablespeakingwithmyimmediate",
    "upper_mgmt": "IfIhadquestionsorconcernsIfeltcomfortablespeakingwithuppermanage",
    "fairness": "Demonstrated2ampequaltreatment",
}

# Synthetic AnyLogic CSV column map. These names come from the AnyLogic export.
STAYER_MAP: Dict[str, str] = {
    "coop_dept": "Cooperation_within_department",
    "coop_other": "Cooperation_with_other_departments",
    "job_training": "Personal_job_training",
    "resources": "Equipment_provided",
    "pmi": "Performance_review_system",
    "training_prog": "Training_programs",
    "pay": "Rate_of_pay",
    "career": "Career_development_opportunities",
    "phys_conditions": "Physical_working_conditions",
    "get_resources": "Ease_of_getting_resources",
    "valued": "Treated_as_valuable_member",
    "sup_feedback": "Supervisor_let_me_know_good_job",
    "sup_suggest": "Free_to_suggest_changes",
    "proper_training": "Proper_training",
    "complaints": "Problems_complaints_resolved",
    "sup_comfort": "Comfortable_speaking_with_supervisor",
    "upper_mgmt": "Comfortable_speaking_with_upper_management",
    "hr_comfort": "Comfortable_speaking_with_HR",
    "informed": "Kept_well_informed",
    "job_security": "Job_security",
    "team": "Team_worked_well_together",
    "safety_culture": "Safety_culture",
    "clarity": "Job_duties_clearly_defined",
    "recognition": "Supervisor_provided_recognition",
    "resolve_problems": "Supervisor_resolved_complaints",
    "sup_sensitive": "Supervisor_sensitive_to_needs",
    "sup_feedback_performance": "Supervisor_feedback_performance",
    "sup_comm": "Supervisor_open_communication",
    "sup_policy": "Supervisor_followed_policies",
    "fairness": "Supervisor_fair_equal_treatment",
    "sup_teamwork": "Supervisor_developed_teamwork",
    "safety_sup": "Supervisor_environment_safety",
}

# Candidate phrases for automatic matching in the Excel leaver file.
LEAVER_CANDIDATES: Dict[str, List[str]] = {
    "coop_dept": ["cooperation within", "department/program", "coop_dept"],
    "coop_other": ["cooperation with other", "other departments", "coop_other"],
    "job_training": ["personal job training", "job training provided", "job_training"],
    "resources": ["equipment", "materials", "resources", "facilities", "equipment provided"],
    "pmi": ["performance review", "pmi"],
    "training_prog": ["training programs", "training_prog"],
    "pay": ["rate of pay", "pay for your job", "pay"],
    "career": ["career development", "advancement opportunities", "career"],
    "phys_conditions": ["physical working conditions", "phys_conditions"],
    "get_resources": ["ability to get", "resources needed", "get resources"],
    "valued": ["valuable member", "treated as a valuable", "valued"],
    "sup_feedback": ["supervisor let me know", "doing a good job"],
    "sup_suggest": ["suggest improvements", "free to suggest"],
    "proper_training": ["proper training", "perform my job effectively"],
    "complaints": ["employee problems", "complaints were resolved promptly", "resolved promptly"],
    "sup_comfort": ["comfortable speaking with my immediate supervisor", "speaking with immediate supervisor"],
    "upper_mgmt": ["upper management"],
    "hr_comfort": ["human resources", "comfortable speaking with hr", "speaking with human resources"],
    "informed": ["kept informed", "well informed", "policies and procedures"],
    "job_security": ["job security"],
    "team": ["team worked well", "members of my team"],
    "safety_culture": ["culture of safety", "supported a culture of safety"],
    "clarity": ["job duties", "responsibilities", "clearly defined"],
    "recognition": ["recognition", "appreciated my work"],
    "resolve_problems": ["complaints and problems were resolved effectively", "problems resolved effectively"],
    "sup_sensitive": ["sensitive to employees", "sensitive to employee"],
    "sup_feedback_performance": ["feedback on performance", "useful feedback"],
    "sup_comm": ["receptive to open communication", "open communication"],
    "sup_policy": ["followed wydot policies", "follows wydot policies"],
    "fairness": ["fair and equal treatment", "fair equal treatment"],
    "sup_teamwork": ["developed cooperation", "develops cooperation", "developed teamwork"],
    "safety_sup": ["environment of safety", "provided an environment of safety"],
}

# -----------------------------------------------------------------------------
# 4. Helpers for loading and scoring data
# -----------------------------------------------------------------------------
def clean_name(x: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()


def score_value(v):
    """Convert text or numeric survey response to 1-4 satisfaction score.

    Meaning: 1 = least favorable/highest concern, 4 = most favorable/lowest concern.
    """
    if pd.isna(v):
        return np.nan
    if isinstance(v, (int, float, np.number)):
        try:
            val = float(v)
            if 1 <= val <= 4:
                return val
            return np.nan
        except Exception:
            return np.nan

    s = clean_name(v)
    mapping = {
        "poor": 1, "fair": 2, "good": 3, "excellent": 4,
        "strongly disagree": 1, "disagree": 2, "agree": 3, "strongly agree": 4,
        "very difficult": 1, "difficult": 2, "easy": 3, "very easy": 4,
        "almost never": 1, "sometimes": 2, "often": 3, "almost always": 4,
        "never": 1, "rarely": 2, "usually": 3, "always": 4,
    }
    if s in mapping:
        return mapping[s]

    # Handle strings like "1 - Poor" or "4=Excellent".
    m = re.match(r"^([1-4])(\D|$)", s)
    if m:
        return float(m.group(1))

    return np.nan


def choose_excel_sheet(path: Path) -> pd.DataFrame:
    sheets = pd.read_excel(path, sheet_name=None)
    # Pick the sheet with the largest number of non-empty rows * columns.
    best_name, best_df, best_score = None, None, -1
    for name, df in sheets.items():
        df2 = df.dropna(how="all").copy()
        score = df2.shape[0] * max(1, df2.shape[1])
        if score > best_score:
            best_name, best_df, best_score = name, df2, score
    print(f"Using leaver Excel sheet: {best_name}  shape={best_df.shape}")
    return best_df


def find_column(df: pd.DataFrame, feature: str, candidates: List[str]) -> str | None:
    # Manual map gets priority.
    if feature in MANUAL_LEAVER_MAP:
        col = MANUAL_LEAVER_MAP[feature]
        if col in df.columns:
            return col
        raise ValueError(f"MANUAL_LEAVER_MAP for {feature} points to missing column: {col}")

    clean_cols = {col: clean_name(col) for col in df.columns}
    cand_clean = [clean_name(c) for c in candidates + [feature]]

    # Exact clean match.
    for col, cc in clean_cols.items():
        if cc in cand_clean:
            return col

    # Candidate phrase contained in column name.
    for cand in cand_clean:
        for col, cc in clean_cols.items():
            if cand and cand in cc:
                return col

    # All tokens in candidate contained in column name.
    for cand in cand_clean:
        tokens = [t for t in cand.split() if len(t) > 2]
        for col, cc in clean_cols.items():
            if tokens and all(t in cc for t in tokens):
                return col

    return None


def standardize_dataframe(df: pd.DataFrame, column_map: Dict[str, str], source_name: str) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    missing = []
    for feat in FEATURES:
        col = column_map.get(feat)
        if col is None or col not in df.columns:
            missing.append(feat)
            out[feat] = np.nan
        else:
            out[feat] = df[col].apply(score_value)

    if missing:
        print(f"WARNING: {source_name}: missing features mapped as NaN: {missing}")

    return out


def build_leaver_map(leaver_df: pd.DataFrame) -> Dict[str, str]:
    out = {}
    missing = []
    for feat in FEATURES:
        col = find_column(leaver_df, feat, LEAVER_CANDIDATES.get(feat, []))
        if col is None:
            missing.append(feat)
        else:
            out[feat] = col

    if missing:
        print("\nThe following leaver features were not automatically found:")
        for feat in missing:
            print(f"  - {feat}")
        print("\nAvailable leaver columns are:")
        for col in leaver_df.columns:
            print(f"  {col}")
        print("\nFix by filling MANUAL_LEAVER_MAP at the top of this script.\n")
        raise ValueError("Automatic leaver column mapping failed. Fill MANUAL_LEAVER_MAP and rerun.")

    pd.DataFrame([{"feature": k, "leaver_column": v} for k, v in out.items()]).to_csv(
        OUTPUT_DIR / "leaver_column_mapping.csv", index=False
    )
    return out


def load_and_build_modeling_data() -> Tuple[pd.DataFrame, pd.Series]:
    if not LEAVER_XLSX.exists():
        raise FileNotFoundError(f"Cannot find {LEAVER_XLSX}")
    if not STAYER_CSV.exists():
        raise FileNotFoundError(f"Cannot find {STAYER_CSV}")

    leaver_raw = choose_excel_sheet(LEAVER_XLSX)
    stayer_raw = pd.read_csv(STAYER_CSV)

    leaver_map = build_leaver_map(leaver_raw)
    leavers = standardize_dataframe(leaver_raw, leaver_map, "leaver")
    stayers = standardize_dataframe(stayer_raw, STAYER_MAP, "stayer")

    leavers["label"] = 1
    stayers["label"] = 0
    leavers["source"] = "real_exit_leaver"
    stayers["source"] = "simulated_stayer"

    combined = pd.concat([leavers, stayers], ignore_index=True)

    # Drop rows with too many missing feature values.
    feature_missing_rate = combined[FEATURES].isna().mean(axis=1)
    before = len(combined)
    combined = combined.loc[feature_missing_rate <= 0.50].copy()
    after = len(combined)
    if after < before:
        print(f"Dropped {before-after} rows with >50% missing survey features.")

    # Report missingness.
    missing_report = combined[FEATURES].isna().mean().sort_values(ascending=False).reset_index()
    missing_report.columns = ["feature", "missing_rate"]
    missing_report.to_csv(OUTPUT_DIR / "feature_missingness.csv", index=False)

    combined.to_csv(OUTPUT_DIR / "combined_modeling_data_32_items.csv", index=False)

    X = combined[FEATURES].copy()
    y = combined["label"].astype(int).copy()
    print(f"\nFinal combined dataset: n={len(combined)}, leavers={int(y.sum())}, stayers={int((1-y).sum())}")
    print(f"Output saved: {OUTPUT_DIR / 'combined_modeling_data_32_items.csv'}")
    return X, y

# -----------------------------------------------------------------------------
# 5. Models and evaluation
# -----------------------------------------------------------------------------
def get_models() -> Dict[str, Pipeline]:
    models: Dict[str, Pipeline] = {}

    models["Logistic Regression"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])

    models["Random Forest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_leaf=3,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    if HAS_XGBOOST:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=300,
                max_depth=2,
                learning_rate=0.05,
                subsample=0.90,
                colsample_bytree=0.90,
                reg_lambda=1.0,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])
    else:
        models["Gradient Boosting (fallback for XGBoost)"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ])

    return models


def evaluate_model(name: str, model: Pipeline, X_train, X_test, y_train, y_test) -> Dict[str, float | str]:
    model.fit(X_train, y_train)
    prob = model.predict_proba(X_test)[:, 1]
    pred = (prob >= 0.50).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()

    return {
        "model": name,
        "test_auc": roc_auc_score(y_test, prob),
        "test_pr_auc": average_precision_score(y_test, prob),
        "test_brier": brier_score_loss(y_test, prob),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred, zero_division=0),
        "recall": recall_score(y_test, pred, zero_division=0),
        "f1": f1_score(y_test, pred, zero_division=0),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def run_model_comparison(X: pd.DataFrame, y: pd.Series) -> Tuple[pd.DataFrame, Dict[str, Pipeline], Dict[str, Pipeline]]:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    models = get_models()
    scoring = {
        "auc": "roc_auc",
        "pr_auc": "average_precision",
        "brier": "neg_brier_score",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
    }
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=RANDOM_STATE)

    rows = []
    fitted_models = {}
    for name, model in models.items():
        print(f"\nEvaluating {name}...")
        cv_res = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        test_res = evaluate_model(name, clone(model), X_train, X_test, y_train, y_test)

        row = dict(test_res)
        row.update({
            "cv_auc_mean": float(np.mean(cv_res["test_auc"])),
            "cv_auc_sd": float(np.std(cv_res["test_auc"])),
            "cv_pr_auc_mean": float(np.mean(cv_res["test_pr_auc"])),
            "cv_brier_mean": float(-np.mean(cv_res["test_brier"])),
            "cv_f1_mean": float(np.mean(cv_res["test_f1"])),
            "cv_recall_mean": float(np.mean(cv_res["test_recall"])),
        })
        rows.append(row)

        fitted = clone(model).fit(X_train, y_train)
        fitted_models[name] = fitted
        joblib.dump(fitted, OUTPUT_DIR / f"{clean_name(name).replace(' ', '_')}_pipeline.pkl")

        save_calibration_plot(name, fitted, X_test, y_test)

    results = pd.DataFrame(rows).sort_values("test_auc", ascending=False)
    results.to_csv(OUTPUT_DIR / "model_comparison_results.csv", index=False)
    print("\nMODEL COMPARISON RESULTS")
    print(results.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print(f"\nSaved: {OUTPUT_DIR / 'model_comparison_results.csv'}")

    return results, fitted_models, {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}


def save_calibration_plot(name: str, model: Pipeline, X_test: pd.DataFrame, y_test: pd.Series) -> None:
    if plt is None:
        return
    prob = model.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, prob, n_bins=6, strategy="uniform")

    plt.figure(figsize=(5, 4))
    plt.plot(mean_pred, frac_pos, marker="o", label=name)
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.xlabel("Predicted probability")
    plt.ylabel("Observed leaver proportion")
    plt.title(f"Calibration: {name}")
    plt.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / f"calibration_{clean_name(name).replace(' ', '_')}.png"
    plt.savefig(path, dpi=200)
    plt.close()

# -----------------------------------------------------------------------------
# 6. Theme scores, synthetic-source detection, and LR artifacts
# -----------------------------------------------------------------------------
def make_theme_features(X: pd.DataFrame) -> pd.DataFrame:
    themes = pd.DataFrame(index=X.index)
    for theme, feats in THEME_GROUPS.items():
        existing = [f for f in feats if f in X.columns]
        themes[theme] = X[existing].mean(axis=1)
    return themes


def centered_features(X: pd.DataFrame) -> pd.DataFrame:
    row_mean = X.mean(axis=1)
    return X.sub(row_mean, axis=0)


def interaction_only_features(X_centered: pd.DataFrame) -> pd.DataFrame:
    imputed = SimpleImputer(strategy="median").fit_transform(X_centered)
    poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
    X_poly = poly.fit_transform(imputed)
    # First n columns are original centered features; remove them to keep interactions only.
    X_int = X_poly[:, X_centered.shape[1]:]
    names = poly.get_feature_names_out(X_centered.columns)[X_centered.shape[1]:]
    return pd.DataFrame(X_int, columns=names, index=X_centered.index)


def run_synthetic_source_detection(X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    """Test whether models can still separate real leavers vs synthetic stayers after stripping raw score levels."""
    feature_sets = {
        "Raw 32 survey items": X.copy(),
        "Eight HR theme means": make_theme_features(X),
        "Within-profile centered items": centered_features(X),
        "Centered interaction-only features": interaction_only_features(centered_features(X)),
    }

    detection_models = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(C=1.0, penalty="l2", solver="liblinear", max_iter=2000, random_state=RANDOM_STATE)),
        ]),
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
    }

    rows = []
    for fs_name, X_fs in feature_sets.items():
        X_train, X_test, y_train, y_test = train_test_split(
            X_fs, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
        )
        for model_name, model in detection_models.items():
            model.fit(X_train, y_train)
            prob = model.predict_proba(X_test)[:, 1]
            rows.append({
                "feature_set": fs_name,
                "model": model_name,
                "auc": roc_auc_score(y_test, prob),
                "pr_auc": average_precision_score(y_test, prob),
                "brier": brier_score_loss(y_test, prob),
            })

    out = pd.DataFrame(rows)
    out.to_csv(OUTPUT_DIR / "synthetic_source_detection_results.csv", index=False)
    print("\nSYNTHETIC-SOURCE DETECTION CHECK")
    print(out.pivot(index="feature_set", columns="model", values="auc").to_string(float_format=lambda x: f"{x:.3f}"))
    print(f"\nSaved: {OUTPUT_DIR / 'synthetic_source_detection_results.csv'}")
    return out

# -----------------------------------------------------------------------------
# 6B. Simulation probability sensitivity analysis
# -----------------------------------------------------------------------------

BASELINE_STAYER_PROBS = {
    "coop_dept": (0.030, 0.100, 0.450, 0.420),
    "coop_other": (0.050, 0.150, 0.500, 0.300),
    "job_training": (0.060, 0.180, 0.500, 0.260),
    "resources": (0.050, 0.150, 0.500, 0.300),
    "pmi": (0.060, 0.180, 0.480, 0.280),
    "training_prog": (0.060, 0.180, 0.500, 0.260),
    "pay": (0.060, 0.180, 0.500, 0.260),
    "career": (0.060, 0.180, 0.480, 0.280),
    "phys_conditions": (0.050, 0.160, 0.520, 0.270),
    "get_resources": (0.040, 0.140, 0.500, 0.320),
    "valued": (0.030, 0.100, 0.450, 0.420),
    "sup_feedback": (0.030, 0.090, 0.420, 0.460),
    "sup_suggest": (0.030, 0.100, 0.450, 0.420),
    "proper_training": (0.050, 0.150, 0.500, 0.300),
    "complaints": (0.050, 0.150, 0.480, 0.320),
    "sup_comfort": (0.030, 0.080, 0.400, 0.490),
    "upper_mgmt": (0.050, 0.150, 0.480, 0.320),
    "hr_comfort": (0.040, 0.140, 0.480, 0.340),
    "informed": (0.040, 0.140, 0.500, 0.320),
    "job_security": (0.030, 0.100, 0.440, 0.430),
    "team": (0.030, 0.100, 0.450, 0.420),
    "safety_culture": (0.030, 0.100, 0.450, 0.420),
    "clarity": (0.040, 0.120, 0.480, 0.360),
    "recognition": (0.030, 0.100, 0.420, 0.450),
    "resolve_problems": (0.030, 0.100, 0.420, 0.450),
    "sup_sensitive": (0.030, 0.090, 0.420, 0.460),
    "sup_feedback_performance": (0.030, 0.100, 0.430, 0.440),
    "sup_comm": (0.030, 0.080, 0.400, 0.490),
    "sup_policy": (0.020, 0.080, 0.400, 0.500),
    "fairness": (0.030, 0.100, 0.420, 0.450),
    "sup_teamwork": (0.030, 0.100, 0.420, 0.450),
    "safety_sup": (0.020, 0.080, 0.400, 0.500),
}


def shift_probability_vector(p, delta):
    """
    Shift a 4-category probability vector.
    Positive delta = more favorable stayer responses, moves probability from 1/2 to 3/4.
    Negative delta = less favorable stayer responses, moves probability from 3/4 to 1/2.
    """
    p = np.array(p, dtype=float).copy()

    if delta > 0:
        move1 = min(p[0], delta / 2)
        move2 = min(p[1], delta / 2)
        moved = move1 + move2
        p[0] -= move1
        p[1] -= move2
        p[2] += moved / 2
        p[3] += moved / 2

    elif delta < 0:
        d = abs(delta)
        move3 = min(p[2], d / 2)
        move4 = min(p[3], d / 2)
        moved = move3 + move4
        p[2] -= move3
        p[3] -= move4
        p[0] += moved / 2
        p[1] += moved / 2

    p = np.clip(p, 0.001, 0.999)
    p = p / p.sum()
    return tuple(p)


def random_perturb_probability_vector(p, rng, scale=0.03):
    """
    Randomly perturb a 4-category probability vector and renormalize.
    """
    p = np.array(p, dtype=float).copy()
    noise = rng.normal(loc=0.0, scale=scale, size=4)
    p = p + noise
    p = np.clip(p, 0.001, None)
    p = p / p.sum()
    return tuple(p)


def make_scenario_probs(scenario_name, seed=42):
    """
    Return modified probability dictionary for one sensitivity scenario.
    """
    rng = np.random.default_rng(seed)
    probs = BASELINE_STAYER_PROBS.copy()

    compensation_items = ["pay", "career", "pmi", "recognition"]
    communication_items = ["informed", "sup_policy", "hr_comfort"]
    safety_items = ["safety_culture", "safety_sup", "job_security"]

    if scenario_name == "baseline":
        return probs

    if scenario_name == "mild_less_favorable_all":
        return {f: shift_probability_vector(p, delta=-0.05) for f, p in probs.items()}

    if scenario_name == "strong_more_favorable_all":
        return {f: shift_probability_vector(p, delta=0.05) for f, p in probs.items()}

    if scenario_name == "compensation_weaker":
        new_probs = probs.copy()
        for f in compensation_items:
            new_probs[f] = shift_probability_vector(probs[f], delta=-0.08)
        return new_probs

    if scenario_name == "communication_safety_stronger":
        new_probs = probs.copy()
        for f in communication_items + safety_items:
            new_probs[f] = shift_probability_vector(probs[f], delta=0.08)
        return new_probs

    if scenario_name == "random_perturbation":
        return {f: random_perturb_probability_vector(p, rng, scale=0.03) for f, p in probs.items()}

    raise ValueError(f"Unknown scenario: {scenario_name}")


def simulate_stayers_numeric(prob_dict, n=300, seed=42):
    """
    Generate numeric synthetic stayer profiles.
    Output uses same 32 FEATURES as the modeling file.
    Scale: 1 = highest concern, 4 = lowest concern.
    """
    rng = np.random.default_rng(seed)
    out = pd.DataFrame(index=range(n))

    for feat in FEATURES:
        p = prob_dict[feat]
        out[feat] = rng.choice([1, 2, 3, 4], size=n, p=p)

    return out


def get_lr_theme_ranking(fitted_lr_pipeline):
    """
    Extract theme ranking from fitted Logistic Regression standardized coefficients.
    """
    lr = fitted_lr_pipeline.named_steps["model"]
    coef = lr.coef_[0]

    rows = []
    for theme, feats in THEME_GROUPS.items():
        idx = [FEATURES.index(f) for f in feats if f in FEATURES]
        rows.append({
            "theme": theme,
            "sum_abs_coef": float(np.sum(np.abs(coef[idx]))),
            "mean_abs_coef": float(np.mean(np.abs(coef[idx]))),
        })

    theme_df = pd.DataFrame(rows).sort_values("sum_abs_coef", ascending=False).reset_index(drop=True)
    theme_df["rank"] = theme_df.index + 1
    return theme_df


def evaluate_one_sensitivity_run(X_leavers, scenario_name, seed):
    """
    Generate one stayer dataset, merge with real leavers, train 3 models, and record results.
    """
    scenario_probs = make_scenario_probs(scenario_name, seed=seed)
    X_stayers = simulate_stayers_numeric(scenario_probs, n=300, seed=seed)

    X_sens = pd.concat([X_leavers[FEATURES], X_stayers[FEATURES]], ignore_index=True)
    y_sens = pd.Series([1] * len(X_leavers) + [0] * len(X_stayers))

    X_train, X_test, y_train, y_test = train_test_split(
        X_sens, y_sens, test_size=TEST_SIZE, stratify=y_sens, random_state=RANDOM_STATE
    )

    rows = []
    models = get_models()

    for model_name, model in models.items():
        fitted = clone(model).fit(X_train, y_train)
        prob = fitted.predict_proba(X_test)[:, 1]
        pred = (prob >= 0.50).astype(int)

        row = {
            "scenario": scenario_name,
            "seed": seed,
            "model": model_name,
            "auc": roc_auc_score(y_test, prob),
            "pr_auc": average_precision_score(y_test, prob),
            "brier": brier_score_loss(y_test, prob),
            "accuracy": accuracy_score(y_test, pred),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred, zero_division=0),
            "f1": f1_score(y_test, pred, zero_division=0),
        }

        if model_name == "Logistic Regression":
            theme_df = get_lr_theme_ranking(fitted)
            top_theme = theme_df.loc[0, "theme"]
            comp_rank = int(theme_df.loc[
                theme_df["theme"] == "Compensation and Career", "rank"
            ].iloc[0])
            row["top_lr_theme"] = top_theme
            row["compensation_rank"] = comp_rank
        else:
            row["top_lr_theme"] = ""
            row["compensation_rank"] = np.nan

        rows.append(row)

    return rows


def run_probability_sensitivity_analysis(X, y, n_repeats=50):
    """
    Main sensitivity analysis.
    Uses real leavers fixed, regenerates synthetic stayers under alternative probability scenarios.
    """
    X_leavers = X.loc[y == 1, FEATURES].copy()

    scenarios = [
        "baseline",
        "mild_less_favorable_all",
        "strong_more_favorable_all",
        "compensation_weaker",
        "communication_safety_stronger",
        "random_perturbation",
    ]

    all_rows = []
    for scenario in scenarios:
        print(f"\nRunning sensitivity scenario: {scenario}")
        for i in range(n_repeats):
            seed = 1000 + i
            all_rows.extend(evaluate_one_sensitivity_run(X_leavers, scenario, seed))

    detailed = pd.DataFrame(all_rows)
    detailed.to_csv(OUTPUT_DIR / "simulation_probability_sensitivity_detailed.csv", index=False)

    summary = detailed.groupby(["scenario", "model"]).agg(
        auc_mean=("auc", "mean"),
        auc_sd=("auc", "std"),
        pr_auc_mean=("pr_auc", "mean"),
        brier_mean=("brier", "mean"),
        accuracy_mean=("accuracy", "mean"),
        recall_mean=("recall", "mean"),
        f1_mean=("f1", "mean"),
    ).reset_index()

    # Logistic Regression theme stability summary
    lr_only = detailed[detailed["model"] == "Logistic Regression"].copy()

    theme_summary = lr_only.groupby("scenario").agg(
        top_theme_mode=("top_lr_theme", lambda x: x.mode().iloc[0] if len(x.mode()) else ""),
        compensation_rank_median=("compensation_rank", "median"),
        compensation_rank_mean=("compensation_rank", "mean"),
        compensation_rank_1_rate=("compensation_rank", lambda x: float(np.mean(x == 1))),
    ).reset_index()

    summary = summary.merge(theme_summary, on="scenario", how="left")
    summary.to_csv(OUTPUT_DIR / "simulation_probability_sensitivity_summary.csv", index=False)

    print("\nSIMULATION PROBABILITY SENSITIVITY SUMMARY")
    print(summary.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
    print(f"\nSaved: {OUTPUT_DIR / 'simulation_probability_sensitivity_detailed.csv'}")
    print(f"Saved: {OUTPUT_DIR / 'simulation_probability_sensitivity_summary.csv'}")

    return detailed, summary
def export_logistic_artifacts(lr_pipeline: Pipeline, X_train: pd.DataFrame) -> None:
    """Export the fitted LR coefficients and scaler values for JavaScript/HTML implementation."""
    imputer: SimpleImputer = lr_pipeline.named_steps["imputer"]
    scaler: StandardScaler = lr_pipeline.named_steps["scaler"]
    lr: LogisticRegression = lr_pipeline.named_steps["model"]

    coef = lr.coef_[0]
    intercept = float(lr.intercept_[0])

    # Theme weights from absolute standardized coefficients.
    coef_df = pd.DataFrame({
        "feature": FEATURES,
        "coefficient_standardized": coef,
        "absolute_coefficient": np.abs(coef),
        "expected_direction_note": "Negative is expected when higher score means more favorable/lower risk",
    })
    coef_df.to_csv(OUTPUT_DIR / "logistic_regression_coefficients.csv", index=False)

    theme_rows = []
    for theme, feats in THEME_GROUPS.items():
        idx = [FEATURES.index(f) for f in feats if f in FEATURES]
        theme_rows.append({
            "theme": theme,
            "sum_abs_standardized_coefficients": float(np.sum(np.abs(coef[idx]))),
            "mean_abs_standardized_coefficients": float(np.mean(np.abs(coef[idx]))),
            "n_items": len(idx),
        })
    theme_df = pd.DataFrame(theme_rows).sort_values("sum_abs_standardized_coefficients", ascending=False)
    theme_df.to_csv(OUTPUT_DIR / "theme_risk_weights_from_lr.csv", index=False)

    artifacts = {
        "model_type": "L2-regularized Logistic Regression",
        "feature_order": FEATURES,
        "imputer_median": {f: float(v) for f, v in zip(FEATURES, imputer.statistics_)},
        "scaler_mean": {f: float(v) for f, v in zip(FEATURES, scaler.mean_)},
        "scaler_scale": {f: float(v) for f, v in zip(FEATURES, scaler.scale_)},
        "coefficients_standardized": {f: float(v) for f, v in zip(FEATURES, coef)},
        "intercept": intercept,
        "theme_groups": THEME_GROUPS,
        "probability_formula": "p = 1/(1 + exp(-(intercept + sum(coef_j * ((x_j_imputed - mean_j)/scale_j)))))",
        "risk_thresholds": {"low": "p < 0.35", "medium": "0.35 <= p < 0.65", "high": "p >= 0.65"},
        "important_warning": "This model is trained on real leavers and simulated stayers. Interpret as proof-of-concept leaver-like separability until validated with real current-employee pulse data.",
    }
    with open(OUTPUT_DIR / "html_logistic_model_artifacts.json", "w", encoding="utf-8") as f:
        json.dump(artifacts, f, indent=2)

    positive_count = int((coef > 0).sum())
    negative_count = int((coef < 0).sum())
    print("\nLOGISTIC REGRESSION COEFFICIENT SIGN CHECK")
    print(f"Positive coefficients: {positive_count}; Negative coefficients: {negative_count}")
    if positive_count > len(FEATURES) * 0.35:
        print("WARNING: Many positive coefficients found. Because 4=more favorable, many coefficients should be negative. Check coding and column mapping.")

    print(f"Saved: {OUTPUT_DIR / 'html_logistic_model_artifacts.json'}")
    print(f"Saved: {OUTPUT_DIR / 'logistic_regression_coefficients.csv'}")
    print(f"Saved: {OUTPUT_DIR / 'theme_risk_weights_from_lr.csv'}")

# -----------------------------------------------------------------------------
# 7. Model selection rule
# -----------------------------------------------------------------------------
def make_model_recommendation(results: pd.DataFrame, detection: pd.DataFrame) -> str:
    best = results.sort_values("test_auc", ascending=False).iloc[0]
    lr = results.loc[results["model"] == "Logistic Regression"].iloc[0]
    best_auc = float(best["test_auc"])
    lr_auc = float(lr["test_auc"])
    best_brier = float(results["test_brier"].min())
    lr_brier = float(lr["test_brier"])

    # Synthetic-source concern: RF on interaction-only features remains high.
    rf_interaction_auc = detection.loc[
        (detection["feature_set"] == "Centered interaction-only features") &
        (detection["model"] == "Random Forest"),
        "auc"
    ].iloc[0]

    recommendation_lines = []
    recommendation_lines.append("MODEL-SELECTION RECOMMENDATION")
    recommendation_lines.append("--------------------------------")
    recommendation_lines.append(f"Best model by held-out AUC: {best['model']} (AUC={best_auc:.3f}).")
    recommendation_lines.append(f"Logistic Regression AUC={lr_auc:.3f}, Brier={lr_brier:.3f}.")
    recommendation_lines.append(f"Random Forest interaction-only synthetic-source AUC={rf_interaction_auc:.3f}.")

    if rf_interaction_auc >= 0.90:
        recommendation_lines.append(
            "High synthetic-source warning: Random Forest can still separate records using interaction-only response structure. Do not deploy RF/XGBoost as the main HR model."
        )

    if (best_auc - lr_auc <= 0.05) and (lr_brier <= best_brier + 0.05):
        recommendation_lines.append(
            "Recommended deployed model for the HTML tool: L2 Logistic Regression, because it is interpretable, calibrated enough for a probability gauge, and close to the best model."
        )
    else:
        recommendation_lines.append(
            "Recommended manuscript decision: report all three models as benchmarks, but do not deploy the highest-AUC black-box model unless it passes synthetic-source and sensitivity checks. For the HR tool, use Logistic Regression or an 8-theme Logistic Regression after checking stability."
        )

    recommendation_lines.append(
        "Use wording: leaver-like risk/separability, not validated turnover prediction, until real current-employee pulse data are available."
    )

    text = "\n".join(recommendation_lines)
    with open(OUTPUT_DIR / "model_selection_recommendation.txt", "w", encoding="utf-8") as f:
        f.write(text)
    print("\n" + text)
    print(f"\nSaved: {OUTPUT_DIR / 'model_selection_recommendation.txt'}")
    return text

# -----------------------------------------------------------------------------
# 8. Main
# -----------------------------------------------------------------------------
def main():
    X, y = load_and_build_modeling_data()
    results, fitted_models, split_data = run_model_comparison(X, y)
    detection = run_synthetic_source_detection(X, y)

    # New sensitivity analysis
    sensitivity_detailed, sensitivity_summary = run_probability_sensitivity_analysis(X, y, n_repeats=50)

    # Export LR artifacts for the HTML tool.
    lr_pipeline = fitted_models["Logistic Regression"]
    export_logistic_artifacts(lr_pipeline, split_data["X_train"])

    make_model_recommendation(results, detection)

    print("\nDONE. Key outputs:")
    for filename in [
        "combined_modeling_data_32_items.csv",
        "model_comparison_results.csv",
        "synthetic_source_detection_results.csv",
        "simulation_probability_sensitivity_detailed.csv",
        "simulation_probability_sensitivity_summary.csv",
        "html_logistic_model_artifacts.json",
        "logistic_regression_coefficients.csv",
        "theme_risk_weights_from_lr.csv",
        "model_selection_recommendation.txt",
    ]:
        print(f"  - {OUTPUT_DIR / filename}")


if __name__ == "__main__":
    main()

Using leaver Excel sheet: Sheet1  shape=(307, 33)

Final combined dataset: n=607, leavers=307, stayers=300
Output saved: /content/wydot_model_outputs/combined_modeling_data_32_items.csv

Evaluating Logistic Regression...

Evaluating Random Forest...

Evaluating XGBoost...

MODEL COMPARISON RESULTS
              model  test_auc  test_pr_auc  test_brier  accuracy  precision  recall    f1  tn  fp  fn  tp  cv_auc_mean  cv_auc_sd  cv_pr_auc_mean  cv_brier_mean  cv_f1_mean  cv_recall_mean
            XGBoost     0.998        0.998       0.017     0.975      0.968   0.984 0.976  58   2   1  61        0.994      0.004           0.995          0.030       0.961           0.941
      Random Forest     0.996        0.997       0.034     0.967      1.000   0.935 0.967  60   0   4  58        0.996      0.003           0.996          0.040       0.951           0.911
Logistic Regression     0.990        0.990       0.045     0.934      0.922   0.952 0.937  55   5   3  59        0.968      0.015     